## 🔧 Step 1: Install Mantis

We begin by installing the `mantis` package directly from GitHub. This will download the latest version of the code and install all required dependencies.

In [1]:
# Clone the full Mantis repository and move into it
!git clone https://github.com/carsondudley1/Mantis.git
%cd Mantis

# Install the package in editable mode
!pip install -e .

Cloning into 'Mantis'...
remote: Enumerating objects: 234, done.
remote: Counting objects: 100% (79/79), done.
remote: Compressing objects: 100% (75/75), done.
remote: Total 234 (delta 38), reused 2 (delta 2), pack-reused 155 (from 1)
Receiving objects: 100% (234/234), 218.61 KiB | 6.07 MiB/s, done.
Resolving deltas: 100% (120/120), done.
/content/Mantis
Obtaining file:///content/Mantis
  Preparing metadata (setup.py) ... done
  Running setup.py develop for mantis


## 📦 Step 2: Download the Pretrained Model

Mantis uses pretrained simulation-grounded foundation models for forecasting.  
In this tutorial, we’ll use the **4-week horizon model with covariates**.

This cell downloads the model file (~1 GB) from the [GitHub release](https://github.com/carsondudley1/Mantis/releases/tag/mantis-v1.0) and places it in a local `models/` directory.

In [2]:
# Download the 4-week model that takes covariates

!mkdir -p models
!wget -O models/mantis_4w_cov.pt https://github.com/carsondudley1/Mantis/releases/download/mantis-v1.0/mantis_4w_cov.pt

--2026-03-18 17:43:48--  https://github.com/carsondudley1/Mantis/releases/download/mantis-v1.0/mantis_4w_cov.pt
Resolving github.com (github.com)... 20.205.243.166
Connecting to github.com (github.com)|20.205.243.166|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/1028419690/2f95c870-6a12-4737-a6f9-a9332d52ae01?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-03-18T18%3A32%3A34Z&rscd=attachment%3B+filename%3Dmantis_4w_cov.pt&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&skt=2026-03-18T17%3A32%3A29Z&ske=2026-03-18T18%3A32%3A34Z&sks=b&skv=2018-11-09&sig=eka45o3iuRHekmb5bBMIFsuzYW6jjoTIhBrCWcXUtWA%3D&jwt=eyJ0eXAiOiJKV1QiLCJhbGciOiJIUzI1NiJ9.eyJpc3MiOiJnaXRodWIuY29tIiwiYXVkIjoicmVsZWFzZS1hc3NldHMuZ2l0aHVidXNlcmNvbnRlbnQuY29tIiwia2V5Ijoia2V5MSIsImV4cCI6MTc3Mzg1OTQyOCwibmJmIjoxNzczODU1ODI4LCJwYXRoIjoicmVsZWFzZWFzc2V0cHJvZ

## 📊 Step 3: Import Mantis and Load Data

We now import the `Mantis` model and load Ebola case and death data. This dataset includes daily case and death counts for Ethiopia.

In [3]:
# Import Mantis and supporting libraries
from mantis import Mantis
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

from google.colab import drive
drive.mount('/content/drive')


ebola_df = pd.read_csv('/content/drive/MyDrive/MantisExperiments/Data/Data_ DRC Ebola Outbreak, North Kivu, Ituri and Équateur - MOH-By-Health-Zone.csv')

ebola_df = ebola_df.iloc[1:]

ebola_df["confirmed_cases"] = pd.to_numeric(ebola_df["confirmed_cases"], errors="coerce")
ebola_df["confirmed_deaths"] = pd.to_numeric(ebola_df["confirmed_deaths"], errors="coerce")

# convert date
ebola_df["report_date"] = pd.to_datetime(ebola_df["report_date"])

# sort properly
ebola_df = ebola_df.sort_values(["health_zone","report_date"])

# compute weekly differences
ebola_df["weekly_cases"] = ebola_df.groupby("health_zone")["confirmed_cases"].diff()
ebola_df["weekly_deaths"] = ebola_df.groupby("health_zone")["confirmed_deaths"].diff()

# replace first NaN values
ebola_df["weekly_cases"] = ebola_df["weekly_cases"].fillna(0)
ebola_df["weekly_deaths"] = ebola_df["weekly_deaths"].fillna(0)

Mounted at /content/drive


## 🤖 Step 4: Initialize Mantis and other models and generate forecasts with metrics

We now create an instance of the Mantis model — specifically, the 4-week horizon version with covariates, as well as our evaluation models.  
Then we pass in the case and death time series to generate forecasts.

Mantis returns 9 quantiles for each of the next 4 weeks, providing a full probabilistic forecast.


In [5]:
import warnings
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.preprocessing import StandardScaler
from tqdm.auto import tqdm
from statsmodels.tsa.api import ETSModel
from statsmodels.tsa.statespace.sarimax import SARIMAX
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

warnings.filterwarnings("ignore")

# ── Config ──────────────────────────────────────────────────────────────────────
FORECAST_HORIZON = 4
STRIDE = 1
START_WEEK = 20
LOOKBACK = 16
LSTM_EPOCHS = 10
LSTM_HIDDEN = 64
LSTM_LAYERS = 2
LSTM_DROPOUT = 0.2
LSTM_BATCH = 32
LSTM_LR = 1e-3

QUANTILES = np.array([0.05, 0.1, 0.25, 0.4, 0.5, 0.6, 0.75, 0.9, 0.95])
MEDIAN_IDX = int(np.where(QUANTILES == 0.5)[0][0])
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ── Mantis model ────────────────────────────────────────────────────────────────
model = Mantis(forecast_horizon=FORECAST_HORIZON, use_covariate=True)

# ── Helper functions ────────────────────────────────────────────────────────────
def make_gaussian_quantiles(mean, stderr, quantiles):
    q = np.zeros((len(mean), len(quantiles)))
    for h in range(len(mean)):
        for j, qq in enumerate(quantiles):
            q[h, j] = mean[h] + stats.norm.ppf(qq) * stderr[h]
    return np.maximum(q, 0.0)

def ensure_monotonic(q_preds):
    out = q_preds.copy()
    for h in range(out.shape[0]):
        out[h, :] = np.sort(out[h, :])
    return out

def compute_window_metrics(true_future, q_preds):
    median = q_preds[:, MEDIAN_IDX]
    model_abs = np.abs(true_future - median)
    cov90_hits = np.sum((true_future >= q_preds[:, 0]) & (true_future <= q_preds[:, 8]))
    cov50_hits = np.sum((true_future >= q_preds[:, 2]) & (true_future <= q_preds[:, 6]))
    return model_abs, int(cov90_hits), int(cov50_hits)

def dm_test(loss1, loss2, h=1):
    d = np.array(loss1) - np.array(loss2)
    if len(d) == 0:
        return np.nan
    mean_d = np.mean(d)
    var_d = np.var(d, ddof=0)
    for k in range(1, h):
        cov_k = np.mean((d[:-k] - mean_d) * (d[k:] - mean_d))
        var_d += 2 * cov_k
    if var_d <= 1e-8:
        return np.nan
    dm_stat = mean_d / np.sqrt(var_d / len(d))
    return 2 * (1 - stats.norm.cdf(abs(dm_stat)))

# ── Forecasting functions ───────────────────────────────────────────────────────
def forecast_mantis(y_hist, x_hist):
    pred = model.predict(
        time_series=y_hist,
        covariate=x_hist,
        target_type=1,
        covariate_type=2,
    )
    return ensure_monotonic(pred)

def forecast_ets(y_hist, x_hist, x_future):
    x_design = np.column_stack([np.ones(len(x_hist)), x_hist])
    beta, *_ = np.linalg.lstsq(x_design, y_hist, rcond=None)
    a, b = float(beta[0]), float(beta[1])
    resid_hist = y_hist - (a + b * x_hist)
    resid_series = pd.Series(resid_hist.astype(float))

    use_seasonal = len(resid_hist) >= 2 * 52 and np.nanstd(resid_hist) > 1e-8
    specs = []
    if use_seasonal:
        specs.append(dict(error="add", trend="add", seasonal="add", seasonal_periods=52))
    specs += [
        dict(error="add", trend="add", seasonal=None),
        dict(error="add", trend=None, seasonal=None),
    ]
    for spec in specs:
        try:
            fit = ETSModel(resid_series, **spec).fit(disp=False)
            pred_obj = fit.get_prediction(
                start=len(resid_hist),
                end=len(resid_hist) + FORECAST_HORIZON - 1,
            )
            resid_mean = np.asarray(pred_obj.predicted_mean, dtype=float)
            in_resid = np.asarray(fit.resid, dtype=float)
            in_resid = in_resid[np.isfinite(in_resid)]
            resid_std = max(np.nanstd(in_resid, ddof=1), 1e-3)
            horizon_se = resid_std * np.sqrt(np.arange(1, FORECAST_HORIZON + 1))
            total_mean = (a + b * x_future) + resid_mean
            return ensure_monotonic(make_gaussian_quantiles(total_mean, horizon_se + 1e-6, QUANTILES))
        except Exception:
            continue
    raise RuntimeError("All ETS specs failed")

def forecast_sarimax(y_hist, x_hist, x_future):
    exog_hist = x_hist.reshape(-1, 1)
    exog_future = x_future.reshape(-1, 1)
    seasonal_order = (1, 0, 0, 52) if len(y_hist) >= 3 * 52 else (0, 0, 0, 0)
    fit = SARIMAX(
        y_hist, exog=exog_hist,
        order=(1, 1, 1), seasonal_order=seasonal_order,
        trend="n", simple_differencing=True,
        enforce_stationarity=False, enforce_invertibility=False,
    ).fit(disp=False, method="lbfgs", maxiter=40)
    fc = fit.get_forecast(steps=FORECAST_HORIZON, exog=exog_future)
    mean = np.asarray(fc.predicted_mean, dtype=float)
    se = np.asarray(fc.se_mean, dtype=float) if hasattr(fc, "se_mean") and fc.se_mean is not None \
        else np.sqrt(np.maximum(np.asarray(fc.var_pred_mean, dtype=float), 1e-8))
    return ensure_monotonic(make_gaussian_quantiles(mean, se + 1e-6, QUANTILES))

# ── LSTM model definition ──────────────────────────────────────────────────────
class QuantileLoss(nn.Module):
    def __init__(self, quantiles):
        super().__init__()
        self.quantiles = torch.tensor(quantiles, dtype=torch.float32)
    def forward(self, preds, target):
        target_exp = target.unsqueeze(-1).expand_as(preds)
        q = self.quantiles.to(preds.device).view(1, 1, -1)
        errors = target_exp - preds
        return torch.max((q - 1) * errors, q * errors).mean()

class LSTMQuantileForecaster(nn.Module):
    def __init__(self, hidden_dim, horizon, n_quantiles, num_layers=2, dropout=0.2):
        super().__init__()
        self.horizon = horizon
        self.n_quantiles = n_quantiles
        self.lstm = nn.LSTM(input_size=2, hidden_size=hidden_dim,
                            num_layers=num_layers,
                            dropout=dropout if num_layers > 1 else 0.0,
                            batch_first=True)
        self.bn = nn.BatchNorm1d(hidden_dim)
        self.head = nn.Sequential(
            nn.Linear(hidden_dim + horizon, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, horizon * n_quantiles),
        )
    def forward(self, x_hist, x_future_cov, use_bn=True):
        out, _ = self.lstm(x_hist)
        h = out[:, -1, :]
        if use_bn and h.shape[0] > 1:
            h = self.bn(h)
        z = torch.cat([h, x_future_cov], dim=1)
        return self.head(z).view(-1, self.horizon, self.n_quantiles)

def forecast_lstm(y_hist, x_hist, x_future):
    n = len(y_hist)
    if n < LOOKBACK + FORECAST_HORIZON + 1:
        raise ValueError("Not enough history for LSTM")
    x_hist_all, x_fc_all, y_all = [], [], []
    for i in range(LOOKBACK, n - FORECAST_HORIZON + 1):
        x_hist_all.append(np.column_stack([y_hist[i-LOOKBACK:i], x_hist[i-LOOKBACK:i]]))
        x_fc_all.append(x_hist[i:i+FORECAST_HORIZON])
        y_all.append(y_hist[i:i+FORECAST_HORIZON])
    x_hist_all = np.asarray(x_hist_all, dtype=float)
    x_fc_all = np.asarray(x_fc_all, dtype=float)
    y_all = np.asarray(y_all, dtype=float)
    if len(x_hist_all) < 2:
        raise ValueError("Too few samples for LSTM")

    sc_h = StandardScaler()
    sc_f = StandardScaler()
    sc_y = StandardScaler()
    x_hist_s = sc_h.fit_transform(x_hist_all.reshape(-1, 2)).reshape(x_hist_all.shape)
    x_fc_s = sc_f.fit_transform(x_fc_all)
    y_s = sc_y.fit_transform(y_all.reshape(-1, 1)).reshape(y_all.shape)

    ds = TensorDataset(
        torch.tensor(x_hist_s, dtype=torch.float32, device=device),
        torch.tensor(x_fc_s, dtype=torch.float32, device=device),
        torch.tensor(y_s, dtype=torch.float32, device=device),
    )
    dl = DataLoader(ds, batch_size=min(LSTM_BATCH, len(ds)), shuffle=True, drop_last=False)

    net = LSTMQuantileForecaster(LSTM_HIDDEN, FORECAST_HORIZON, len(QUANTILES),
                                  LSTM_LAYERS, LSTM_DROPOUT).to(device)
    opt = torch.optim.Adam(net.parameters(), lr=LSTM_LR)
    loss_fn = QuantileLoss(QUANTILES)
    net.train()
    for _ in range(LSTM_EPOCHS):
        for xb_h, xb_f, yb in dl:
            opt.zero_grad()
            loss_fn(net(xb_h, xb_f, use_bn=True), yb).backward()
            opt.step()

    hist_now = np.column_stack([y_hist[-LOOKBACK:], x_hist[-LOOKBACK:]])
    fut_now = x_future.reshape(1, -1)
    hist_t = torch.tensor(sc_h.transform(hist_now).reshape(1, LOOKBACK, 2),
                          dtype=torch.float32, device=device)
    fut_t = torch.tensor(sc_f.transform(fut_now), dtype=torch.float32, device=device)
    net.eval()
    with torch.no_grad():
        q_scaled = net(hist_t, fut_t, use_bn=False).squeeze(0).cpu().numpy()
    q = np.zeros_like(q_scaled)
    for qi in range(q.shape[1]):
        q[:, qi] = sc_y.inverse_transform(q_scaled[:, qi].reshape(-1, 1)).ravel()
    return ensure_monotonic(np.maximum(q, 0.0))

# ── Main evaluation: loop over ALL provinces ─────────────────────────────────────
methods = ["mantis", "ets_x", "sarimax_x", "lstm_q"]
provinces = sorted(ebola_df["province"].dropna().unique())

print(f"Found {len(provinces)} provinces in dataset")
print(f"Target: weekly_cases | Covariate: weekly_deaths")
print(f"Config: horizon={FORECAST_HORIZON}, stride={STRIDE}, start_week={START_WEEK}")
print(f"Filter: skipping provinces where >50% of case counts are 0\n")

all_province_summaries = []
skipped_provinces = []
pooled_errors = {m: [] for m in methods}

for province in tqdm(provinces, desc="Provinces"):
    province_df = ebola_df[ebola_df["province"] == province].sort_values("report_date")
    province_ts = province_df.groupby("report_date")[["weekly_cases", "weekly_deaths"]].sum().reset_index()

    cases_ts = province_ts["weekly_cases"].fillna(0).values.astype(float)
    deaths_ts = province_ts["weekly_deaths"].fillna(0).values.astype(float)

    # Clamp negatives to 0
    cases_ts[cases_ts < 0] = 0
    deaths_ts[deaths_ts < 0] = 0

    # target = cases, covariate = deaths
    y_full = cases_ts
    x_full = deaths_ts

    # Skip provinces where more than half of case counts are under 5
    if np.mean(y_full == 0) > 0.5:
        skipped_provinces.append((province, f"low cases ({np.sum(y_full > 0)}/{len(y_full)} weeks > 0)"))
        continue

    # Skip provinces that are too short
    if len(y_full) < START_WEEK + FORECAST_HORIZON + 1:
        skipped_provinces.append((province, f"too short ({len(y_full)} weeks)"))
        continue

    stats_store = {
        m: {"model_abs": [], "cov90_hits": 0, "cov50_hits": 0,
            "total_points": 0, "fallbacks": 0}
        for m in methods
    }

    forecast_weeks = list(range(START_WEEK, len(y_full) - FORECAST_HORIZON + 1, STRIDE))
    if len(forecast_weeks) == 0:
        skipped_provinces.append((province, f"no forecast windows ({len(y_full)} weeks)"))
        continue

    for t in tqdm(forecast_weeks, desc=f"  {province}", leave=False):
        y_hist = y_full[:t]
        x_hist = x_full[:t]
        y_true = y_full[t:t + FORECAST_HORIZON]
        if len(y_true) < FORECAST_HORIZON:
            continue

        x_future = np.repeat(x_hist[-1], FORECAST_HORIZON)

        forecasters = {
            "mantis":    lambda: forecast_mantis(y_hist, x_hist),
            "ets_x":     lambda: forecast_ets(y_hist, x_hist, x_future),
            "sarimax_x": lambda: forecast_sarimax(y_hist, x_hist, x_future),
            "lstm_q":    lambda: forecast_lstm(y_hist, x_hist, x_future),
        }

        for mname, fn in forecasters.items():
            try:
                q_pred = fn()
            except Exception:
                stats_store[mname]["fallbacks"] += 1
                q_pred = np.tile(
                    np.repeat(y_hist[-1], FORECAST_HORIZON).reshape(-1, 1),
                    (1, len(QUANTILES)),
                )

            m_abs, c90, c50 = compute_window_metrics(y_true, q_pred)
            s = stats_store[mname]
            s["model_abs"].extend(m_abs)
            s["cov90_hits"] += c90
            s["cov50_hits"] += c50
            s["total_points"] += FORECAST_HORIZON

    # Build per-province summary
    mantis_errors = stats_store["mantis"]["model_abs"]
    for mname in methods:
        s = stats_store[mname]
        pooled_errors[mname].extend(s["model_abs"])
        mae = float(np.mean(s["model_abs"])) if s["model_abs"] else np.nan
        cov90 = s["cov90_hits"] / s["total_points"] if s["total_points"] > 0 else np.nan
        cov50 = s["cov50_hits"] / s["total_points"] if s["total_points"] > 0 else np.nan
        dm_p = np.nan if mname == "mantis" else dm_test(mantis_errors, s["model_abs"], h=FORECAST_HORIZON)
        all_province_summaries.append({
            "province": province, "method": mname, "mae": mae,
            "coverage_50": cov50, "coverage_90": cov90,
            "dm_p_value": dm_p, "fallbacks": s["fallbacks"],
            "forecast_points": s["total_points"],
        })

# ── Results ─────────────────────────────────────────────────────────────────────
results_df = pd.DataFrame(all_province_summaries)

print(f"\n{'='*70}")
print(f"Skipped {len(skipped_provinces)} provinces:")
for name, reason in skipped_provinces:
    print(f"  {name}: {reason}")

# Per-province tables
evaluated_provinces = results_df["province"].unique()
print(f"\nEvaluated {len(evaluated_provinces)} provinces\n")

for p in evaluated_provinces:
    pdf = results_df[results_df["province"] == p].sort_values("mae")
    print(f"\n--- {p} ---")
    print(pdf.drop(columns="province").to_string(index=False, float_format=lambda x: f"{x:.4f}"))

# Aggregate across all provinces
print(f"\n{'='*70}")
print("=== AGGREGATE (mean across all provinces) ===\n")
agg_df = results_df.groupby("method").agg(
    mae=("mae", "mean"),
    coverage_50=("coverage_50", "mean"),
    coverage_90=("coverage_90", "mean"),
    total_fallbacks=("fallbacks", "sum"),
    total_forecast_points=("forecast_points", "sum"),
).sort_values("mae").reset_index()
print(agg_df.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

# Aggregate DM test (pooled errors across all provinces)
print("\n=== AGGREGATE Diebold-Mariano (pooled errors across all provinces) ===\n")
for mname in methods:
    if mname == "mantis":
        continue
    p = dm_test(pooled_errors["mantis"], pooled_errors[mname], h=FORECAST_HORIZON)
    print(f"  Mantis vs {mname}: p = {p:.4f}")

Found 3 provinces in dataset
Target: weekly_cases | Covariate: weekly_deaths
Config: horizon=4, stride=1, start_week=20
Filter: skipping provinces where >50% of case counts are 0



Provinces:   0%|          | 0/3 [00:00<?, ?it/s]

  North Kivu:   0%|          | 0/553 [00:00<?, ?it/s]


Skipped 2 provinces:
  Ituri: low cases (225/576 weeks > 0)
  South Kivu: low cases (4/209 weeks > 0)

Evaluated 1 provinces


--- North Kivu ---
   method    mae  coverage_50  coverage_90  dm_p_value  fallbacks  forecast_points
   mantis 2.7423       0.5023       0.9096         NaN          0             2212
    ets_x 3.1048       0.5633       0.8486      0.0000          0             2212
   lstm_q 3.4126       0.0692       0.1867      0.0000          1             2212
sarimax_x 4.7647       0.4177       0.6519      0.0000          0             2212

=== AGGREGATE (mean across all provinces) ===

   method    mae  coverage_50  coverage_90  total_fallbacks  total_forecast_points
   mantis 2.7423       0.5023       0.9096                0                   2212
    ets_x 3.1048       0.5633       0.8486                0                   2212
   lstm_q 3.4126       0.0692       0.1867                1                   2212
sarimax_x 4.7647       0.4177       0.6519                